#   E. coli wavelet analysis template

##  Description

This notebook is a reusable starting point for wavelet-based analysis of the *E. coli* genome. It reuses the RefSeq loading workflow and binary-signal construction logic from the EDA notebook, then provides helper functions for preparing wavelet artifacts for use in downstream analyses.

The intent is to make it easy to build signals such as GC, A, C, G, or T over the full genome, CDS regions, or non-CDS regions, and then prepare approximation/detail coefficients, reconstructed components, etc etc.

##  Goal

Characterize how variability in GC and A/T/C/G signals is distributed across scales and locations, and to determine whether coding and non-coding regions exhibit distinguishable multiscale signatures. Ultimately, we would like to find interesting structures and patterns, and potentially artifacts that do not show up in traditional signal processing or global analysis tools like Fourier transformations.

##  Possible TODOs
- Compare wavelet approximation and detail coefficients across full-genome, CDS, and non-CDS signals.
- Study energy distribution across scales and how it changes across signal type or genomic region.
- Build spatial plots of reconstructed detail coefficients across scales.
- Investigate entropy across scales and relative entropy or divergence between CDS and non-CDS profiles.
- Add windowed wavelet comparisons (A vs C, or GC vs C, for instance).
- Add error analysis/quantification.
- Analyze computational complexity and scaling efficiency of wavelet operation.


In [1]:
import io
import math

import numpy as np
import pandas as pd
import pywt
from Bio import Entrez, SeqIO
import Bio

ACCESSION = "NC_000913.3"  # E. coli K-12 MG1655 complete genome
EMAIL = "daniel_mcgonigle1@student.uml.edu"  # NCBI asks for an email

Entrez.email = EMAIL


In [2]:
def fetch_genbank_record(accession: str) -> Bio.SeqRecord.SeqRecord:
    """
    Fetch a nucleotide sequence record from NCBI and parse it into a Biopython SeqRecord.

    Parameters
    ----------
    accession : str
        RefSeq or GenBank accession for the genome of interest.

    Returns
    -------
    Bio.SeqRecord.SeqRecord
        Parsed record containing the DNA sequence and feature annotations.
    """
    with Entrez.efetch(db="nuccore", id=accession, rettype="gbwithparts", retmode="text") as handle:
        text = handle.read()
    return SeqIO.read(io.StringIO(text), "genbank")


record = fetch_genbank_record(ACCESSION)
sequence = str(record.seq).upper()
genome_len = len(sequence)

CDS_FEATURES = {"CDS"}
NONCDS_FEATURES = {feat.type for feat in record.features} - CDS_FEATURES

print(f"Record ID: {record.id}")
print(f"Description: {record.description}")
print(f"Genome length: {genome_len:,} bp")
print(f"Annotated feature types: {sorted({feat.type for feat in record.features})}")


Record ID: NC_000913.3
Description: Escherichia coli str. K-12 substr. MG1655, complete genome
Genome length: 4,641,652 bp
Annotated feature types: ['CDS', 'gene', 'misc_feature', 'mobile_element', 'ncRNA', 'rRNA', 'rep_origin', 'source', 'tRNA']


In [3]:
def extract_feature_sequence(
    sequence: str,
    record: Bio.SeqRecord.SeqRecord,
    feature_types: set[str],
) -> str:
    """
    Extract and concatenate only the sequence covered by selected feature types.

    Parameters
    ----------
    sequence : str
        Full nucleotide sequence.
    record : Bio.SeqRecord.SeqRecord
        Record whose feature annotations determine which regions are kept.
    feature_types : set[str]
        Annotation types to retain, such as {"CDS"} or NONCDS_FEATURES.

    Returns
    -------
    str
        Concatenated subsequence formed from all matching feature intervals,
        preserving the order in which those features appear in the record.

    Notes
    -----
    Compound feature locations such as joins are expanded part-by-part.
    """
    if not feature_types:
        raise ValueError("feature_types must contain at least one feature type")

    segments = []
    for feat in record.features:
        if feat.type not in feature_types:
            continue

        parts = getattr(feat.location, "parts", [feat.location])
        for part in parts:
            start = int(part.start)
            end = int(part.end)
            segments.append(sequence[start:end])

    return "".join(segments)


def create_binary_signal(
    bases: str,
    sequence: str,
    record: Bio.SeqRecord.SeqRecord | None = None,
    feature_types: set[str] | None = None,
) -> np.ndarray:
    """
    Create a binary nucleotide indicator signal.

    Parameters
    ----------
    bases : str
        One or more nucleotide letters to mark with 1s, for example "GC" or "A".
    sequence : str
        Full sequence from which the signal is built.
    record : Bio.SeqRecord.SeqRecord | None, optional
        Required when feature_types is provided so the corresponding regions can
        be extracted from the annotated genome.
    feature_types : set[str] | None, optional
        If provided, the function first condenses the sequence down to only the
        selected feature types, then builds the binary signal on that reduced
        sequence.

    Returns
    -------
    np.ndarray
        One-dimensional int8 array of 0s and 1s.
    """
    selected_bases = {base.upper() for base in bases if not base.isspace()}
    if not selected_bases:
        raise ValueError("bases must contain at least one nucleotide letter")

    if feature_types is None:
        sequence_to_encode = sequence
    else:
        if record is None:
            raise ValueError("record must be provided when feature_types is specified")
        sequence_to_encode = extract_feature_sequence(sequence, record, feature_types)

    return np.fromiter(
        (1 if base in selected_bases else 0 for base in sequence_to_encode),
        dtype=np.int8,
    )


In [4]:
def max_wavelet_level(signal_length: int, wavelet: str | pywt.Wavelet) -> int:
    """
    Compute the largest admissible discrete wavelet decomposition level.
    """
    if signal_length <= 0:
        raise ValueError("signal_length must be positive")

    wavelet_obj = pywt.Wavelet(wavelet) if isinstance(wavelet, str) else wavelet
    return pywt.dwt_max_level(signal_length, wavelet_obj.dec_len)


def compute_wavelet_decomposition(
    signal: np.ndarray,
    wavelet: str | pywt.Wavelet = "haar",
    level: int | None = None,
    mode: str = "periodization",
) -> dict:
    """
    Compute a multilevel one-dimensional discrete wavelet decomposition.

    Parameters
    ----------
    signal : np.ndarray
        One-dimensional input signal.
    wavelet : str | pywt.Wavelet, default "haar"
        Wavelet family to use.
    level : int | None, optional
        Number of decomposition levels. If omitted, the maximum useful level is used.
    mode : str, default "periodization"
        Boundary-handling mode passed to PyWavelets.

    Returns
    -------
    dict
        Structured decomposition metadata including the original signal,
        approximation coefficients, and detail coefficients keyed by scale.
    """
    x = np.asarray(signal, dtype=float)
    if x.ndim != 1:
        raise ValueError("signal must be one-dimensional")
    if len(x) == 0:
        raise ValueError("signal must not be empty")

    wavelet_obj = pywt.Wavelet(wavelet) if isinstance(wavelet, str) else wavelet
    admissible_level = max_wavelet_level(len(x), wavelet_obj)
    use_level = admissible_level if level is None else min(level, admissible_level)
    if use_level < 1:
        raise ValueError("signal is too short for a multilevel decomposition")

    coeffs = pywt.wavedec(x, wavelet=wavelet_obj, level=use_level, mode=mode)
    detail_coeffs = {
        detail_level: coeffs[-detail_level]
        for detail_level in range(1, use_level + 1)
    }

    return {
        "signal": x,
        "signal_length": len(x),
        "wavelet": wavelet_obj,
        "mode": mode,
        "level": use_level,
        "coeffs": coeffs,
        "approximation": coeffs[0],
        "details": detail_coeffs,
    }


def reconstruct_multiscale_components(decomposition: dict) -> dict[str, dict[int, np.ndarray]]:
    """
    Reconstruct approximation and detail components back to signal length.

    Returns
    -------
    dict[str, dict[int, np.ndarray]]
        Dictionary with two keys:
        - "approximations": reconstructed approximation component at each level
        - "details": reconstructed detail component at each level

        Each reconstructed array is trimmed to the original signal length so it can
        be plotted directly against genome position or local index.
    """
    coeffs = decomposition["coeffs"]
    wavelet = decomposition["wavelet"]
    mode = decomposition["mode"]
    level = decomposition["level"]
    signal_length = decomposition["signal_length"]

    approximations = {}
    details = {}

    for target_level in range(1, level + 1):
        approx_coeffs = [np.zeros_like(arr) for arr in coeffs]
        detail_coeffs = [np.zeros_like(arr) for arr in coeffs]

        approx_coeffs[0] = coeffs[0].copy()
        for idx in range(1, level - target_level + 1):
            approx_coeffs[idx] = coeffs[idx].copy()

        detail_index = level - target_level + 1
        detail_coeffs[detail_index] = coeffs[detail_index].copy()

        approximations[target_level] = pywt.waverec(approx_coeffs, wavelet, mode=mode)[:signal_length]
        details[target_level] = pywt.waverec(detail_coeffs, wavelet, mode=mode)[:signal_length]

    return {"approximations": approximations, "details": details}


In [5]:
def summarize_wavelet_coefficients(decomposition: dict) -> pd.DataFrame:
    """
    Create a tidy summary table for approximation and detail coefficients.

    This is useful for checking coefficient counts, scale levels, and raw energies
    before building plots or comparisons.
    """
    rows = []

    approx = np.asarray(decomposition["approximation"])
    rows.append({
        "kind": "approximation",
        "level": decomposition["level"],
        "num_coeffs": len(approx),
        "mean": float(np.mean(approx)),
        "std": float(np.std(approx)),
        "energy": float(np.sum(np.square(approx))),
    })

    for level, coeffs in sorted(decomposition["details"].items()):
        detail = np.asarray(coeffs)
        rows.append({
            "kind": "detail",
            "level": level,
            "num_coeffs": len(detail),
            "mean": float(np.mean(detail)),
            "std": float(np.std(detail)),
            "energy": float(np.sum(np.square(detail))),
        })

    return pd.DataFrame(rows).sort_values(["kind", "level"]).reset_index(drop=True)


def compute_wavelet_energy(
    decomposition: dict,
    normalize: bool = True,
    include_approximation: bool = True,
) -> pd.DataFrame:
    """
    Compute coefficient energy by wavelet component.

    Parameters
    ----------
    decomposition : dict
        Output from compute_wavelet_decomposition.
    normalize : bool, default True
        If True, divide each component energy by the total included energy.
    include_approximation : bool, default True
        If True, include the final approximation coefficients in the output table.

    Returns
    -------
    pd.DataFrame
        Table that supports scale-energy plots, normalized comparisons, and
        entropy calculations.
    """
    rows = []

    if include_approximation:
        approx = np.asarray(decomposition["approximation"])
        rows.append({
            "kind": "approximation",
            "level": decomposition["level"],
            "energy": float(np.sum(np.square(approx))),
        })

    for level, coeffs in sorted(decomposition["details"].items()):
        detail = np.asarray(coeffs)
        rows.append({
            "kind": "detail",
            "level": level,
            "energy": float(np.sum(np.square(detail))),
        })

    energy_df = pd.DataFrame(rows).sort_values(["kind", "level"]).reset_index(drop=True)
    if normalize:
        total_energy = energy_df["energy"].sum()
        energy_df["relative_energy"] = energy_df["energy"] / total_energy if total_energy else 0.0

    return energy_df


def compute_scale_entropy(energy_df: pd.DataFrame, column: str = "relative_energy") -> float:
    """
    Compute Shannon entropy for an energy distribution across scales.
    """
    if column not in energy_df.columns:
        raise ValueError(f"column '{column}' not found in energy_df")

    probs = energy_df[column].to_numpy(dtype=float)
    probs = probs[probs > 0]
    if len(probs) == 0:
        return 0.0
    return float(-np.sum(probs * np.log2(probs)))


def compare_energy_profiles(
    energy_df_a: pd.DataFrame,
    energy_df_b: pd.DataFrame,
    column: str = "relative_energy",
    metric: str = "kl",
    epsilon: float = 1e-12,
) -> float:
    """
    Compare two scale-energy profiles.

    Supported metrics
    -----------------
    "kl"
        Kullback-Leibler divergence D_KL(a || b).
    "js"
        Jensen-Shannon divergence.
    "l2"
        Euclidean distance between profiles.
    """
    merged = pd.merge(
        energy_df_a[["kind", "level", column]],
        energy_df_b[["kind", "level", column]],
        on=["kind", "level"],
        how="inner",
        suffixes=("_a", "_b"),
    )
    if merged.empty:
        raise ValueError("energy profiles do not share any common components")

    a = merged[f"{column}_a"].to_numpy(dtype=float) + epsilon
    b = merged[f"{column}_b"].to_numpy(dtype=float) + epsilon
    a = a / a.sum()
    b = b / b.sum()

    if metric == "kl":
        return float(np.sum(a * np.log2(a / b)))
    if metric == "js":
        m = 0.5 * (a + b)
        return float(0.5 * np.sum(a * np.log2(a / m)) + 0.5 * np.sum(b * np.log2(b / m)))
    if metric == "l2":
        return float(np.linalg.norm(a - b))
    raise ValueError("metric must be one of {'kl', 'js', 'l2'}")


In [6]:
def generate_signal_windows(
    signal: np.ndarray,
    window_size: int,
    step: int | None = None,
    drop_incomplete: bool = False,
) -> list[dict]:
    """
    Split a signal into overlapping or non-overlapping windows.

    Returns
    -------
    list[dict]
        Each item stores window bounds and the corresponding signal slice so it can
        be passed into wavelet helper functions for localized analysis.
    """
    x = np.asarray(signal)
    if x.ndim != 1:
        raise ValueError("signal must be one-dimensional")
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    step = window_size if step is None else step
    if step <= 0:
        raise ValueError("step must be positive")

    windows = []
    for start in range(0, len(x), step):
        end = start + window_size
        window = x[start:end]
        if len(window) < window_size and drop_incomplete:
            break
        if len(window) == 0:
            continue
        windows.append({
            "start": start,
            "end": min(end, len(x)),
            "signal": window,
        })
    return windows


def compute_windowed_wavelet_energy(
    signal: np.ndarray,
    window_size: int,
    step: int | None = None,
    wavelet: str | pywt.Wavelet = "haar",
    level: int | None = None,
    mode: str = "periodization",
    normalize: bool = True,
    include_approximation: bool = False,
) -> pd.DataFrame:
    """
    Compute scale-energy summaries over sliding windows.

    This prepares a tidy table suitable for heat maps, windowed line plots, and
    regional comparisons without prescribing a single visualization.
    """
    rows = []
    for window in generate_signal_windows(signal, window_size=window_size, step=step):
        decomposition = compute_wavelet_decomposition(
            window["signal"],
            wavelet=wavelet,
            level=level,
            mode=mode,
        )
        energy_df = compute_wavelet_energy(
            decomposition,
            normalize=normalize,
            include_approximation=include_approximation,
        )
        energy_df["window_start"] = window["start"]
        energy_df["window_end"] = window["end"]
        rows.append(energy_df)

    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)


def prepare_wavelet_artifacts(
    signal: np.ndarray,
    wavelet: str | pywt.Wavelet = "haar",
    level: int | None = None,
    mode: str = "periodization",
    normalize_energy: bool = True,
    include_approximation_energy: bool = True,
    reconstruct_components: bool = True,
) -> dict:
    """
    Prepare a bundle of reusable wavelet artifacts for downstream analysis.

    Parameters
    ----------
    signal : np.ndarray
        One-dimensional signal to analyze.
    wavelet : str | pywt.Wavelet, default "haar"
        Wavelet family to use.
    level : int | None, optional
        Number of decomposition levels.
    mode : str, default "periodization"
        Boundary-handling mode passed to PyWavelets.
    normalize_energy : bool, default True
        Whether to include normalized energy in the energy table.
    include_approximation_energy : bool, default True
        Whether to include final approximation energy in the energy summary.
    reconstruct_components : bool, default True
        Whether to reconstruct approximation and detail components back to signal length.

    Returns
    -------
    dict
        Dictionary containing reusable wavelet-analysis artifacts with the following keys:

        - "decomposition": raw structured output from compute_wavelet_decomposition(...).
          This includes the original signal, selected wavelet, decomposition level,
          full coefficient list, final approximation coefficients, and detail
          coefficients keyed by level.
        - "coefficient_summary": pandas DataFrame with one row per approximation
          or detail component and columns such as coefficient count, mean,
          standard deviation, and coefficient energy.
        - "energy_summary": pandas DataFrame with scale-wise energy values and,
          when requested, normalized relative energy values for comparisons across
          scales or across different signals.
        - "reconstructions": optional dictionary returned only when
          reconstruct_components=True. It contains reconstructed approximation and
          detail components mapped back to signal length so they can be plotted or
          compared spatially.

        The goal is to return one bundle that can support coefficient inspection,
        energy plots, entropy calculations, and localized spatial analyses.
    """
    decomposition = compute_wavelet_decomposition(
        signal,
        wavelet=wavelet,
        level=level,
        mode=mode,
    )
    artifacts = {
        "decomposition": decomposition,
        "coefficient_summary": summarize_wavelet_coefficients(decomposition),
        "energy_summary": compute_wavelet_energy(
            decomposition,
            normalize=normalize_energy,
            include_approximation=include_approximation_energy,
        ),
    }
    if reconstruct_components:
        artifacts["reconstructions"] = reconstruct_multiscale_components(decomposition)
    return artifacts


def estimate_dwt_operation_counts(
    signal_length: int,
    wavelet: str | pywt.Wavelet = "haar",
    level: int | None = None,
) -> pd.DataFrame:
    """
    Estimate per-level coefficient counts as a lightweight proxy for DWT work.

    This is not a full runtime model, but it gives a structured starting point for
    computational-complexity discussion.
    """
    if signal_length <= 0:
        raise ValueError("signal_length must be positive")

    wavelet_obj = pywt.Wavelet(wavelet) if isinstance(wavelet, str) else wavelet
    admissible_level = max_wavelet_level(signal_length, wavelet_obj)
    use_level = admissible_level if level is None else min(level, admissible_level)

    rows = []
    current_length = signal_length
    for j in range(1, use_level + 1):
        next_length = math.ceil(current_length / 2)
        rows.append({
            "level": j,
            "input_length": current_length,
            "approx_length": next_length,
            "detail_length": next_length,
            "filter_length": wavelet_obj.dec_len,
            "rough_convolution_work": current_length * wavelet_obj.dec_len,
        })
        current_length = next_length

    return pd.DataFrame(rows)


## Example usage

The cells below give a lightweight example of how to build signals, prepare wavelet artifacts, and inspect a few key summaries. They are meant as a starting point rather than a complete analysis.


In [7]:
gc_signal = create_binary_signal("GC", sequence, record=record)
gc_cds_signal = create_binary_signal("GC", sequence, record=record, feature_types=CDS_FEATURES)
gc_noncds_signal = create_binary_signal("GC", sequence, record=record, feature_types=NONCDS_FEATURES)

print("Full GC signal length:", len(gc_signal))
print("CDS GC signal length:", len(gc_cds_signal))
print("Non-CDS GC signal length:", len(gc_noncds_signal))
print("Full GC fraction:", gc_signal.mean())
print("CDS GC fraction:", gc_cds_signal.mean())
print("Non-CDS GC fraction:", gc_noncds_signal.mean())


Full GC signal length: 4641652
CDS GC signal length: 4026887
Non-CDS GC signal length: 9118190
Full GC fraction: 0.5079070985933456
CDS GC fraction: 0.5189845158307149
Non-CDS GC fraction: 0.5120894607372735


In [8]:
gc_artifacts = prepare_wavelet_artifacts(
    gc_signal,
    wavelet="haar",
    level=6,
    reconstruct_components=False,
)

print("Wavelet used:", gc_artifacts["decomposition"]["wavelet"].name)
print("Decomposition level:", gc_artifacts["decomposition"]["level"])
print("Approximation coefficient count:", len(gc_artifacts["decomposition"]["approximation"]))
print("Detail levels:", list(gc_artifacts["decomposition"]["details"].keys()))

gc_artifacts["coefficient_summary"].head()


Wavelet used: haar
Decomposition level: 6
Approximation coefficient count: 72526
Detail levels: [1, 2, 3, 4, 5, 6]


,kind,level,num_coeffs,mean,std,energy
0,approximation,6,72526,4.063251,0.661482,1.229140e+06
1,detail,1,2320826,-0.000016,0.483315,5.421300e+05
2,detail,2,1160413,-0.000032,0.500130,2.902540e+05
3,detail,3,580207,-0.000003,0.531435,1.638636e+05
4,detail,4,290104,-0.000371,0.517327,7.763981e+04


In [9]:
gc_energy = gc_artifacts["energy_summary"]
gc_windowed_energy = compute_windowed_wavelet_energy(
    gc_signal,
    window_size=4096,
    step=2048,
    wavelet="haar",
    level=5,
)

print("Scale entropy:", compute_scale_entropy(gc_energy))
print("Number of windowed energy rows:", len(gc_windowed_energy))
print("Windowed energy columns:", list(gc_windowed_energy.columns))

gc_energy


Scale entropy: 1.925828401156311
Number of windowed energy rows: 11335
Windowed energy columns: ['kind', 'level', 'energy', 'relative_energy', 'window_start', 'window_end']


,kind,level,energy,relative_energy
0,approximation,6,1.229140e+06,0.521368
1,detail,1,5.421300e+05,0.229957
2,detail,2,2.902540e+05,0.123118
3,detail,3,1.638636e+05,0.069507
4,detail,4,7.763981e+04,0.032933
5,detail,5,3.671553e+04,0.015574
6,detail,6,1.778627e+04,0.007544


##  Usage Notes
- Detail level is analogous to frequency.
- If we want a rough comparison to Fourier ideas, detail level `D1` captures the finest-scale oscillations (0.25 to 0.5 for Haar).
- Squaring the coefficients is analogous to energy in a Fourier power spectrum.
- `prepare_wavelet_artifacts(...)` returns one dictionary so you can compute once and reuse the results throughout the notebook.
    - `artifacts["decomposition"]` contains the raw wavelet metadata and coefficient objects, including the approximation coefficients and detail coefficients by level.
    - `artifacts["coefficient_summary"]` is the quickest table for inspecting counts, means, standard deviations, and energies of the approximation/detail components.
    - `artifacts["energy_summary"]` is the main table for scale-energy plots, normalized energy comparisons, and entropy-based analysis.
    - `artifacts["reconstructions"]` is only present when `reconstruct_components=True`; it stores approximation/detail components reconstructed back to signal length for spatial plotting and localized interpretation.

##  Useful Variable Usage
- `artifacts["decomposition"]["approximation"]` gives the final approximation coefficients at the deepest decomposition level.
- `artifacts["decomposition"]["details"][j]` gives the raw detail coefficients for detail level `j`.
- `artifacts["reconstructions"]["approximations"][j]` and `artifacts["reconstructions"]["details"][j]` give those components reconstructed back to the original signal length.
- `artifacts["decomposition"]["coeffs"]` preserves the original PyWavelets `wavedec(...)` output exactly as returned.
- Read `coeffs` as `[cA_n, cD_n, cD_{n-1}, ..., cD_1]`, where `cA_n` is the final approximation coefficient array and the remaining entries are detail coefficients from coarse to fine scale.
- The `details` dictionary is a friendlier reorganization of the detail part of `coeffs`, so you can ask for `details[2]` directly instead of manually indexing the PyWavelets list.


In [16]:
# Get Detail Level 2 coefficients
gc_artifacts["decomposition"]["details"][2]

array([-4.26642159e-17, -5.00000000e-01, -5.00000000e-01, ...,
       -5.00000000e-01,  0.00000000e+00, -5.00000000e-01],
      shape=(1160413,))

In [17]:
# Get approximation coefficients
gc_artifacts["decomposition"]["approximation"]

array([3.125, 2.875, 2.875, ..., 4.25 , 4.25 , 1.875], shape=(72526,))

In [19]:
# Get reconstructed level 2 approximations (requires reconstruct_components=True)
# gc_artifacts["reconstructions"]["approximations"][2]